In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import os
print("Done")

Done


In [ ]:
import os

# Create dataset directory
os.makedirs("/content/datasets", exist_ok=True)
os.chdir("/content/datasets")

# Download VisDrone using ultralytics built-in
from ultralytics.utils.downloads import download

# VisDrone DET dataset
urls = [
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
]

for url in urls:
    download(url, dir="/content/datasets")

print("Download complete")

WARNING ⚠️ Skipping /content/datasets/VisDrone2019-DET-train.zip unzip as destination directory /content/datasets/VisDrone2019-DET-train is not empty.
WARNING ⚠️ Skipping /content/datasets/VisDrone2019-DET-val.zip unzip as destination directory /content/datasets/VisDrone2019-DET-val is not empty.
Download complete


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

def visdrone2yolo(dir):
    """Convert VisDrone annotations to YOLO format."""

    # VisDrone classes
    # 0=ignored, 1=pedestrian, 2=people, 3=bicycle, 4=car, 5=van,
    # 6=truck, 7=tricycle, 8=awning-tricycle, 9=bus, 10=motor, 11=others

    # We map to our simplified classes (skip 0 and 11)
    dir = Path(dir)
    (dir / 'labels').mkdir(parents=True, exist_ok=True)

    pbar = (dir / 'annotations').glob('*.txt')
    for f in pbar:
        img_file = dir / 'images' / f.stem
        # Find image with any extension
        for ext in ['.jpg', '.jpeg', '.png']:
            if (img_file.with_suffix(ext)).exists():
                img_file = img_file.with_suffix(ext)
                break

        # Get image dimensions
        try:
            im = Image.open(img_file)
            w, h = im.size
        except:
            continue

        lines = []
        with open(f) as ann:
            for row in ann.readlines():
                vals = row.strip().split(',')
                if len(vals) < 8:
                    continue

                cls = int(vals[5])
                if cls == 0 or cls == 11:  # skip ignored and others
                    continue

                # Convert to 0-indexed and remap
                cls = cls - 1  # now 0=pedestrian, 1=people, 2=bicycle...

                # Bounding box: x, y, w, h in pixels
                x, y, bw, bh = int(vals[0]), int(vals[1]), int(vals[2]), int(vals[3])

                # Convert to YOLO format (normalized center x, y, w, h)
                xc = (x + bw / 2) / w
                yc = (y + bh / 2) / h
                bw_n = bw / w
                bh_n = bh / h

                if bw_n > 0 and bh_n > 0:
                    lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw_n:.6f} {bh_n:.6f}")

        # Write label file
        with open(dir / 'labels' / f"{f.stem}.txt", 'w') as lf:
            lf.write('\n'.join(lines))

    print(f"Converted annotations in {dir}")

# Convert both train and val
visdrone2yolo("/content/datasets/VisDrone2019-DET-train")
visdrone2yolo("/content/datasets/VisDrone2019-DET-val")
print("Conversion complete")


Converted annotations in /content/datasets/VisDrone2019-DET-train
Converted annotations in /content/datasets/VisDrone2019-DET-val
Conversion complete


In [ ]:
import yaml

dataset_config = {
    'path': '/content/datasets',
    'train': 'VisDrone2019-DET-train/images',
    'val': 'VisDrone2019-DET-val/images',
    'nc': 10,
    'names': [
        'pedestrian', 'people', 'bicycle', 'car', 'van',
        'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
    ]
}

with open('/content/visdrone.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("Dataset config created")
print(yaml.dump(dataset_config))

Dataset config created
names:
- pedestrian
- people
- bicycle
- car
- van
- truck
- tricycle
- awning-tricycle
- bus
- motor
nc: 10
path: /content/datasets
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images



In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8n (COCO weights - this is our baseline)
model = YOLO('yolov8n.pt')

# Fine-tune on VisDrone
results = model.train(
    data='/content/visdrone.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,           # GPU
    project='/content/uav_ifs_runs',
    name='yolov8n_visdrone',
    pretrained=True,
    verbose=True
)

print("Training complete")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_visdrone, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

In [ ]:
from google.colab import files
files.download('/content/uav_ifs_runs/yolov8n_visdrone/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>